# ICU Resource Forecasting (Expanded Metrics)
This notebook forecasts ICU bed usage, ventilator demand, and nurse staffing requirements using Prophet.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from prophet import Prophet
import gradio as gr
import datetime

%matplotlib inline

## 1. Simulate Historical ICU Data for Multiple Metrics

In [ ]:
# Simulate historical time series data
np.random.seed(42)
dates = pd.date_range(start='2022-01-01', periods=200)
icu_data = pd.DataFrame({'date': dates})
icu_data['icu_beds'] = np.random.poisson(20, 200) + np.sin(np.linspace(0, 6.28, 200)) * 5
icu_data['ventilators'] = np.random.poisson(10, 200) + np.cos(np.linspace(0, 6.28, 200)) * 3
icu_data['nurses'] = icu_data['icu_beds'] * 0.5 + np.random.normal(0, 1, 200)
icu_data.head()

## 2. Create and Train Forecasting Models for Each Metric

In [ ]:
# Create separate models for each resource
models = {}
forecasts = {}
for metric in ['icu_beds', 'ventilators', 'nurses']:
    df = icu_data[['date', metric]].rename(columns={'date': 'ds', metric: 'y'})
    model = Prophet()
    model.fit(df)
    future = model.make_future_dataframe(periods=30)
    forecast = model.predict(future)
    models[metric] = model
    forecasts[metric] = forecast

## 3. Visualize Forecasts

In [ ]:
for metric, model in models.items():
    print(f'Forecast for {metric.replace("_", " ").title()}')
    model.plot(forecasts[metric]); plt.show()

## 4. Gradio App for Interactive Forecasting by Metric

In [ ]:
def forecast_resource(metric, days):
    df = icu_data[['date', metric]].rename(columns={'date': 'ds', metric: 'y'})
    model = Prophet()
    model.fit(df)
    future = model.make_future_dataframe(periods=days)
    forecast = model.predict(future)
    fig = model.plot(forecast)
    return fig

gr.Interface(
    fn=forecast_resource,
    inputs=[gr.Radio(['icu_beds', 'ventilators', 'nurses'], label='Metric'),
            gr.Slider(7, 60, value=30, label='Days to Forecast')],
    outputs=gr.Plot(),
    title='ICU Forecasting by Resource',
    description='Choose a resource metric to forecast ICU needs.'
).launch()